# Notebook 02 — Data Quality

**Purpose:** Diagnose every data quality problem in the raw dataset.  
**Rule:** We only *observe* here — no fixes yet. Fixes happen in Notebook 03.

Think of this notebook as a doctor's diagnosis report. We are listing all the diseases before deciding on treatment.

---

## What we check in this notebook

| Issue | What it means | Why it matters |
|---|---|---|
| Missing values | A cell is empty (NaN) | Missing eligibility text → can't match users |
| Duplicate rows | Same scheme appears twice | Inflates recommendations for that scheme |
| Useless columns | Column has no analytical value | Wastes memory, confuses the pipeline |
| Text noise | HTML tags, BOM chars, extra spaces | Corrupts TF-IDF scores |
| Inconsistent categories | 'Women & Child' vs 'Women and Child' | Breaks grouping and filtering |

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 300)
pd.set_option('display.max_rows', 60)

print('Libraries loaded.')

In [ ]:
# ── Load raw data ─────────────────────────────────────────────────────────
RAW_PATH = '../data/raw/updated_data.csv'
df = pd.read_csv(RAW_PATH)

print(f'Loaded: {df.shape[0]} rows × {df.shape[1]} columns')

---
## Issue 1 — Missing Values

**`NaN`** = Not a Number. In pandas, this is how a missing/empty cell is represented.

`df.isnull()` returns a DataFrame of True/False where True = the cell is empty.  
`df.isnull().sum()` counts the True values per column = count of missing cells.

In [ ]:
# ── 1.1 Count missing values per column ──────────────────────────────────
null_counts = df.isnull().sum()
null_pct    = (df.isnull().sum() / len(df) * 100).round(2)

missing_report = pd.DataFrame({
    'missing_count': null_counts,
    'missing_pct'  : null_pct,
    'present_count': len(df) - null_counts,
}).sort_values('missing_count', ascending=False)

print('Missing value report:')
print('─' * 55)
print(missing_report.to_string())

In [ ]:
# ── 1.2 Show sample rows where key text fields are missing ────────────────
key_text_cols = ['details', 'benefits', 'eligibility', 'application', 'documents']

for col in key_text_cols:
    missing_mask = df[col].isnull()
    n_missing    = missing_mask.sum()
    if n_missing > 0:
        print(f'\n── {col}: {n_missing} missing rows ──')
        sample = df[missing_mask][['scheme_name', 'level', col]].head(3)
        print(sample.to_string())
    else:
        print(f'\n── {col}: No missing values ✓')

**Decision guide for missing text fields:**

| Field | If missing → do this | Why |
|---|---|---|
| `details` | Fill with empty string `''` | Missing details means no TF-IDF signal from that field — not fatal |
| `benefits` | Fill with empty string `''` | Same reasoning |
| `eligibility` | Fill with empty string `''` | Eligibility engine will score it as 'Unknown' on all criteria |
| `application` | Fill with empty string `''` | Not used in NLP pipeline — just display |
| `documents` | Fill with empty string `''` | Same — display only |
| `tags` | Fill with empty string `''` | Minor impact on TF-IDF |
| `schemeCategory` | Fill with `'Uncategorized'` | Needed for display and filtering |
| `level` | Fill with `'Unknown'` | Needed for eligibility engine |
| `slug` | Fill with slugified `scheme_name` | Needed as URL identifier |

---
## Issue 2 — Duplicate Rows

**`df.duplicated()`** marks rows that are exact copies of a previous row as True.

We check duplicates on:
1. `scheme_name` alone — exact name match
2. `slug` alone — if the slug (URL identifier) is the same, it's definitely the same scheme
3. Both together — most reliable check

In [ ]:
# ── 2.1 Exact full-row duplicates ────────────────────────────────────────
full_dupes = df.duplicated().sum()
print(f'Exact full-row duplicates: {full_dupes}')

# ── 2.2 Duplicates on scheme_name ────────────────────────────────────────
name_dupes = df.duplicated(subset=['scheme_name']).sum()
print(f'Duplicate scheme_name values: {name_dupes}')

# ── 2.3 Duplicates on slug ───────────────────────────────────────────────
slug_dupes = df.duplicated(subset=['slug']).sum()
print(f'Duplicate slug values: {slug_dupes}')

In [ ]:
# ── 2.4 Show the actual duplicate records ────────────────────────────────
if name_dupes > 0:
    duped_names = df[df.duplicated(subset=['scheme_name'], keep=False)]
    print(f'All rows that have a duplicated scheme_name ({len(duped_names)} rows):')
    print(duped_names[['scheme_name', 'slug', 'level', 'schemeCategory']]
          .sort_values('scheme_name').head(20).to_string())
else:
    print('No scheme_name duplicates found.')

**Decision:** Remove exact duplicates (keep first occurrence). For schemes with the same name but slightly different content, inspect them manually before deciding.

---
## Issue 3 — Useless Columns

We already spotted one: the unnamed empty column between `schemeCategory` and `tags`.
Let's confirm it is 100% empty and identify any other dead-weight columns.

In [ ]:
# ── 3.1 Completely empty columns ─────────────────────────────────────────
print('Columns with ALL values missing:')
for col in df.columns:
    if df[col].isnull().all():
        print(f'  {repr(col)} → 100% empty → DROP')

print()

# ── 3.2 Nearly empty columns (>95% missing) ──────────────────────────────
print('Columns with >95% values missing:')
for col in df.columns:
    pct = df[col].isnull().mean() * 100
    if pct > 95:
        print(f'  {repr(col)} → {pct:.1f}% missing → consider DROP')

In [ ]:
# ── 3.3 Evaluate the `slug` column ───────────────────────────────────────
# slug = a URL-friendly version of the scheme name
# It's a good unique identifier but has no analytical value for NLP.
# We keep it as a database key.
print('Slug column — sample values:')
print(df['slug'].head(10).to_list())
print(f'Unique slugs: {df["slug"].nunique()} / {len(df)} rows')

---
## Issue 4 — Text Noise

The data was scraped from websites. Website text often contains:
- **BOM characters** (`\ufeff` or ``) — invisible characters at the start of a file/field
- **`\n`, `\r`** — newline characters that should be spaces in our context
- **`&amp;`, `&nbsp;`, `&lt;`** — HTML entities (e.g., `&amp;` is the escaped version of `&`)
- **Extra whitespace** — double spaces, trailing spaces

These corrupt TF-IDF: `"scheme"` and `"scheme\n"` become different tokens!

In [ ]:
# ── 4.1 Check for BOM characters ─────────────────────────────────────────
bom_char = '\ufeff'

print('BOM character (\\ufeff) occurrences per text column:')
text_cols = ['scheme_name', 'details', 'benefits', 'eligibility', 'application', 'documents', 'tags']
for col in text_cols:
    count = df[col].dropna().str.contains(bom_char, regex=False).sum()
    if count > 0:
        print(f'  {col}: {count} rows contain BOM chars')

In [ ]:
# ── 4.2 Check for HTML entities ───────────────────────────────────────────
html_pattern = r'&[a-zA-Z]+;|&#\d+;'

print('HTML entity occurrences per text column:')
for col in text_cols:
    count = df[col].dropna().str.contains(html_pattern, regex=True).sum()
    if count > 0:
        # Show a sample
        sample_row = df[df[col].str.contains(html_pattern, regex=True, na=False)].iloc[0]
        # Find the entity in the text
        found = re.findall(html_pattern, str(sample_row[col]))[:5]
        print(f'  {col}: {count} rows | sample entities: {found}')

In [ ]:
# ── 4.3 Check for excessive whitespace and newlines ───────────────────────
print('Rows with 3+ consecutive newlines in `details` (first 5):')
excessive = df['details'].dropna().str.contains(r'\n{3,}', regex=True)
print(f'  Count: {excessive.sum()}')

print()
print('Rows with leading/trailing whitespace in `scheme_name` (first 5):')
leading = df['scheme_name'].dropna().str.match(r'^\s|\s$')
print(f'  Count: {leading.sum()}')

In [ ]:
# ── 4.4 Show a raw text sample to see what we're working with ─────────────
print('Raw `details` field of row 0 (first 500 chars):')
print(repr(df['details'].iloc[0][:500]))

---
## Issue 5 — Category Inconsistency

Categories may have spelling variations, extra spaces, or mixed separators  
(some use `&`, some use `and`). This matters for filtering and display.

In [ ]:
# ── 5.1 All unique schemeCategory values ──────────────────────────────────
cats = df['schemeCategory'].dropna().unique()
cats.sort()
print(f'Total unique categories: {len(cats)}')
print()
for c in cats:
    print(f'  {repr(c)}')

In [ ]:
# ── 5.2 Look for categories that appear to be the same but spelled differently
# Normalize: strip, lowercase, then compare
norm_cats = df['schemeCategory'].dropna().str.strip().str.lower().unique()
norm_cats.sort()
print(f'Unique categories after normalization: {len(norm_cats)}')
print('(If this number is lower than above, there are case/space inconsistencies)')

---
## Issue 6 — Scheme Name Noise

Some scheme names appear to be wrapped in extra quotes (from the CSV scrape).  
Example: `'"Immediate Relief Assistance" under "Welfare..."'`  
These leading/trailing quotes need to be stripped.

In [ ]:
# ── 6.1 Schemes with quote-wrapped names ──────────────────────────────────
quoted = df['scheme_name'].str.startswith('"').fillna(False)
print(f'Scheme names starting with a quote character: {quoted.sum()}')
if quoted.sum() > 0:
    print('Samples:')
    print(df[quoted]['scheme_name'].head(5).to_list())

---
## Summary — Full Quality Report

Run all cells above, then fill this table with your findings:

| Issue | Finding | Fix in Notebook 03 |
|---|---|---|
| Unnamed empty column | 100% empty | Drop it |
| Missing `details` | _(count from cell 1.1)_ | Fill with `''` |
| Missing `benefits` | _(count)_ | Fill with `''` |
| Missing `eligibility` | _(count)_ | Fill with `''` |
| Missing `tags` | _(count)_ | Fill with `''` |
| Missing `schemeCategory` | _(count)_ | Fill with `'Uncategorized'` |
| Full-row duplicates | _(count)_ | Drop (keep first) |
| BOM characters | _(count)_ | Strip in text cleaning |
| HTML entities | _(count)_ | Replace with decoded form |
| Excessive newlines | _(count)_ | Normalize to single space |
| Quoted scheme names | _(count)_ | Strip surrounding quotes |
| Category inconsistency | _(count)_ | Strip + title-case |

**Next step:** Notebook 03 will fix every issue in this table.